# Imports

In [33]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json
import re
import string
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score
from pprint import pp
import numpy as np
from tqdm import tqdm
from datetime import datetime
import time

# Load reranker

In [34]:
def load_reranker(model_path_or_name: str = "ncbi/MedCPT-Cross-Encoder", device: str = None):
    """
    Loads the MedCPT cross-encoder model and its tokenizer.

    Args:
        model_path_or_name : HuggingFace model name or path to a local finetuned checkpoint.
        device             : Target device string ("cpu", "cuda", "cuda:1", ...).
                             If None, defaults to cuda if available, else cpu.

    Returns:
        model     : The loaded model in eval mode, on the target device.
        tokenizer : The associated tokenizer.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_path_or_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_path_or_name)
    model.to(device)
    model.eval()

    print(f"Reranker loaded from '{model_path_or_name}' on device '{device}'.")
    return model, tokenizer

In [35]:
model, tokenizer = load_reranker()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9940.16it/s]

Reranker loaded from 'ncbi/MedCPT-Cross-Encoder' on device 'cpu'.


# Load Dataset

In [36]:
def load_bioASQ(
    abstracts_path: str,
    bioasq_path: str,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42
) -> tuple[Dataset, Dataset, Dataset, dict]:
    """
    Loads and preprocesses the BioASQ dataset and the tokenized abstracts.

    Args:
        abstracts_path : Path to the abstract_list_tokenized.json file.
        bioasq_path    : Path to the BioASQ training JSON file.
        train_ratio    : Fraction of data for training.
        val_ratio      : Fraction of data for validation.
        test_ratio     : Fraction of data for testing.
        seed           : Random seed for reproducibility.

    Returns:
        train_dataset  : HF Dataset for training.
        val_dataset    : HF Dataset for validation.
        test_dataset   : HF Dataset for testing.
        abstracts      : Raw dict mapping PubMed ID (str) to list of sentences.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, \
        "train_ratio, val_ratio and test_ratio must sum to 1."

    with open(abstracts_path, 'r', encoding='utf-8') as f:
        abstracts = json.load(f)

    with open(bioasq_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    questions = raw['questions']

    entries = []
    for q in questions:
        pubmed_ids = [
            match[0]
            for link in q.get('documents', [])
            for match in [re.findall(r"pubmed\/(\d+)", link)]
            if match
        ]

        # Filter to IDs that are actually present in the tokenized abstracts
        available_ids = [pid for pid in pubmed_ids if pid in abstracts]

        if not available_ids:
            continue

        snippets = [
            {
                "document_id": re.findall(r"pubmed\/(\d+)", s["document"])[0],
                "start": s["offsetInBeginSection"],
                "end": s["offsetInEndSection"],
                "text": s["text"]
            }
            for s in q.get("snippets", [])
            if re.findall(r"pubmed\/(\d+)", s["document"])
        ]

        entries.append({
            "id": q["id"],
            "query": q["body"],
            "pubmed_ids": available_ids,
            "snippets": snippets
        })

    # First split off the test set, then split remainder into train/val
    test_size = test_ratio
    val_size = val_ratio / (train_ratio + val_ratio)

    train_val, test = train_test_split(entries, test_size=test_size, random_state=seed)
    train, val = train_test_split(train_val, test_size=val_size, random_state=seed)

    train_dataset = Dataset.from_list(train)
    val_dataset = Dataset.from_list(val)
    test_dataset = Dataset.from_list(test)

    print(f"Dataset loaded: {len(train_dataset)} train / {len(val_dataset)} val / {len(test_dataset)} test entries.")
    print(f"Entries dropped (no matching abstracts): {len(questions) - len(entries)}")

    return train_dataset, val_dataset, test_dataset, abstracts

In [37]:
train_dataset, val_dataset, test_dataset, abstracts = load_bioASQ(
    abstracts_path='../data/BioASQ-training14b/abstract_list_tokenized.json',
    bioasq_path='../data/BioASQ-training14b/training14b.json',
)



Dataset loaded: 4582 train / 573 val / 573 test entries.
Entries dropped (no matching abstracts): 1


In [38]:
pp(train_dataset[0])

{'id': '603219c21cb411341a000133',
 'query': 'What is vesiduction?',
 'pubmed_ids': ['32363801', '32784449'],
 'snippets': [{'document_id': '32363801',
               'end': 338,
               'start': 0,
               'text': 'Besides the canonical gene transfer mechanisms '
                       'transformation, transduction and conjugation, DNA '
                       'transfer involving extracellular vesicles is still '
                       'under appreciated. However, this widespread phenomenon '
                       'has been observed in the three domains of life. Here, '
                       "we propose the term 'Vesiduction' as a fourth mode of "
                       'intercellular DNA transfer.'},
              {'document_id': '32784449',
               'end': 1028,
               'start': 885,
               'text': 'In this review, we first summarize the major pathways '
                       'of HGT in bacteria, including conjugation, '
                       '

# Prepare candidates

In this section, we want, beforehand, to preprare the dataset in a format compatible for testing. 

In [39]:
def normalize(text: str) -> list[str]:
    """
    Lowercases, strips punctuation, collapses whitespace, and splits into words.
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()


def lcs_length(words_a: list[str], words_b: list[str]) -> int:
    """
    Computes the length of the Longest Common Substring between two word lists.
    """
    n, m = len(words_a), len(words_b)
    best = 0
    # dp[i][j] = length of longest common substring ending at words_a[i-1], words_b[j-1]
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if words_a[i - 1] == words_b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
                best = max(best, dp[i][j])
            else:
                dp[i][j] = 0
    return best


def compute_label(sentence_words: list[str], snippets: list[dict]) -> float:
    """
    Returns a float relevance score in [0, 1] representing the best overlap
    between the sentence and any snippet in the query.

    Score is computed as max(lcs / len(snippet_words), lcs / len(sentence_words))
    across all snippets, taking the maximum across snippets.

    Args:
        sentence_words : Normalized word list of the candidate sentence.
        snippets       : List of snippet dicts with a 'text' key (pre-filtered
                         to available abstracts upstream).

    Returns:
        Float in [0, 1]. Returns 0.0 if no snippets are available.
    """
    best = 0.0
    for snippet in snippets:
        snippet_words = normalize(snippet['text'])
        if not snippet_words:
            continue
        lcs = lcs_length(sentence_words, snippet_words)
        score = max(lcs / len(snippet_words), lcs / len(sentence_words))
        best = max(best, score)
    return best


def build_candidate_pool(
    entry: dict,
    abstracts: dict,
) -> dict:
    """
    Builds the candidate pool for a single query entry.

    Args:
        entry     : A single dataset entry with keys 'id', 'query', 'pubmed_ids', 'snippets'.
        abstracts : Dict mapping PubMed ID (str) to list of sentence strings.
        threshold : Minimum fraction of snippet covered by LCS to assign label 1.

    Returns:
        A dict with:
            'id'         : Query ID.
            'query'      : Query string.
            'candidates' : List of {'sentence': str, 'label': int}.
    """
    # Filter snippets to those whose source abstract is available
    valid_snippets = [
        s for s in entry['snippets']
        if s['document_id'] in abstracts
    ]

    candidates = []
    for pid in entry['pubmed_ids']:
        if pid not in abstracts:
            continue
        for sentence in abstracts[pid]:
            sentence_words = normalize(sentence)
            if not sentence_words:
                continue
            label = compute_label(sentence_words, valid_snippets)
            candidates.append({
                'sentence': sentence,
                'relevance_score': label
            })

    return {
        'id': entry['id'],
        'query': entry['query'],
        'candidates': candidates
    }

In [ ]:
# train_pool = [build_candidate_pool(entry, abstracts) for entry in train_dataset]

KeyboardInterrupt: 

In [ ]:
# test_pool = [build_candidate_pool(entry, abstracts) for entry in test_dataset]

KeyboardInterrupt: 

In [ ]:
# with open('train_pool.json', 'w', encoding='utf-8') as f:
#     json.dump(train_pool, f, ensure_ascii=False, indent=2)

# with open('test_pool.json', 'w', encoding='utf-8') as f:
#     json.dump(test_pool, f, ensure_ascii=False, indent=2)

NameError: name 'train_pool' is not defined

In [41]:
with open('test_pool.json', 'r', encoding='utf-8') as f:
    test_pool = json.load(f)

# Rerank

In [ ]:
def rerank(
    query: str,
    candidates: list[dict],
    model,
    tokenizer,
    batch_size: int = 16,
    device: str = None
) -> list[dict]:
    """
    Scores and sorts candidates by relevance to the query using the cross-encoder.

    Args:
        query      : The query string.
        candidates : List of dicts with at least 'sentence' and 'relevance_score' keys.
        model      : Loaded cross-encoder model.
        tokenizer  : Associated tokenizer.
        batch_size : Number of (query, sentence) pairs per forward pass.
        device     : Target device. If None, inferred from model parameters.

    Returns:
        Candidates sorted by predicted relevance score descending,
        each dict extended with a 'predicted_score' key.
    """
    if device is None:
        device = next(model.parameters()).device

    sentences = [c['sentence'] for c in candidates]
    all_scores = []

    model.eval()
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sentences = sentences[i:i + batch_size]
            encoded = tokenizer(
                [[query, s] for s in batch_sentences],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            ).to(device)
            logits = model(**encoded).logits.squeeze(-1)
            all_scores.extend(logits.cpu().tolist())

    ranked = [
        {**c, 'predicted_score': score}
        for c, score in zip(candidates, all_scores)
    ]
    ranked.sort(key=lambda x: x['predicted_score'], reverse=True)

    return ranked

In [42]:
# alt rerank with time and token perf
def rerank(
    query: str,
    candidates: list[dict],
    model,
    tokenizer,
    batch_size: int = 16,
    device: str = None
) -> tuple[list[dict], dict]:
    """
    Scores and sorts candidates by relevance to the query using the cross-encoder.

    Args:
        query      : The query string.
        candidates : List of dicts with at least 'sentence' and 'relevance_score' keys.
        model      : Loaded cross-encoder model.
        tokenizer  : Associated tokenizer.
        batch_size : Number of (query, sentence) pairs per forward pass.
        device     : Target device. If None, inferred from model parameters.

    Returns:
        ranked   : Candidates sorted by predicted relevance score descending,
                   each dict extended with a 'predicted_score' key.
        metadata : Dict with reranker_time_ms and reranker_input_tokens.
    """
    if device is None:
        device = next(model.parameters()).device

    sentences = [c['sentence'] for c in candidates]
    all_scores = []
    total_tokens = 0

    model.eval()
    start = time.perf_counter()

    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sentences = sentences[i:i + batch_size]
            encoded = tokenizer(
                [[query, s] for s in batch_sentences],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            ).to(device)
            total_tokens += encoded['input_ids'].numel()
            logits = model(**encoded).logits.squeeze(-1)
            all_scores.extend(logits.cpu().tolist())

    elapsed_ms = (time.perf_counter() - start) * 1000

    ranked = [
        {**c, 'predicted_score': score}
        for c, score in zip(candidates, all_scores)
    ]
    ranked.sort(key=lambda x: x['predicted_score'], reverse=True)

    metadata = {
        "reranker_time_ms": elapsed_ms,
        "reranker_input_tokens": total_tokens
    }

    return ranked, metadata

In [43]:
query = test_pool[1]['query']
candidate = test_pool[1]['candidates']

scoreTest1, metadata1 = rerank(query=query, candidates=candidate, model=model, tokenizer=tokenizer)

query = test_pool[2]['query']
candidate = test_pool[2]['candidates']

scoreTest2, metadata2 = rerank(query=query, candidates=candidate, model=model, tokenizer=tokenizer)

# Evaluation of the reranker

In [44]:
def evaluate(
    ranked_candidates: list[dict],
    threshold: float = 0.5,
    k_values: list[int] = [5, 10, 20]
) -> dict:
    """
    Evaluates a single query's ranked candidate list against binary relevance labels.

    Args:
        ranked_candidates : Output of rerank() — list of dicts with 'relevance_score'
                            and 'predicted_score', sorted by predicted_score descending.
        threshold         : Cutoff for binarizing relevance_score into a label.
        k_values          : List of cutoff values for @k metrics.

    Returns:
        Dict of metric names to float values.
    """
    labels = [int(c['relevance_score'] >= threshold) for c in ranked_candidates]
    scores = [c['predicted_score'] for c in ranked_candidates]

    n_relevant = sum(labels)
    results = {}

    # MAP
    num_hits = 0
    sum_precision = 0.0
    for i, label in enumerate(labels):
        if label == 1:
            num_hits += 1
            sum_precision += num_hits / (i + 1)
    results['MAP'] = (sum_precision / n_relevant) if n_relevant > 0 else 0.0

    # MRR
    results['MRR'] = 0.0
    for i, label in enumerate(labels):
        if label == 1:
            results['MRR'] = 1.0 / (i + 1)
            break

    # NDCG@k, P@k, R@k, F1@k
    for k in k_values:
        top_k_labels = labels[:k]
        n_relevant_at_k = sum(top_k_labels)

        # NDCG@k — sklearn expects 2D arrays
        if n_relevant > 0:
            ndcg = ndcg_score(
                y_true=np.array([labels]),
                y_score=np.array([scores]),
                k=k
            )
        else:
            ndcg = 0.0
        results[f'NDCG@{k}'] = ndcg

        # P@k
        precision = n_relevant_at_k / k
        results[f'P@{k}'] = precision

        # R@k
        recall = (n_relevant_at_k / n_relevant) if n_relevant > 0 else 0.0
        results[f'R@{k}'] = recall

        # F1@k
        if precision + recall > 0:
            results[f'F1@{k}'] = 2 * precision * recall / (precision + recall)
        else:
            results[f'F1@{k}'] = 0.0

    return results


def evaluate_reranker_dataset(
    all_ranked_candidates: list[list[dict]],
    all_metadata: list[dict],
    threshold: float = 0.5,
    k_values: list[int] = [5, 10, 20]
) -> dict:

    all_results = [
        evaluate(ranked, threshold, k_values)
        for ranked in all_ranked_candidates
    ]

    averaged = {}
    for metric in all_results[0].keys():
        averaged[metric] = float(np.mean([r[metric] for r in all_results]))

    averaged['avg_reranker_time_ms'] = float(np.mean([m['reranker_time_ms'] for m in all_metadata]))
    averaged['avg_reranker_input_tokens'] = float(np.mean([m['reranker_input_tokens'] for m in all_metadata]))

    return averaged

In [46]:
pp(evaluate_reranker_dataset([scoreTest1, scoreTest2], all_metadata=[metadata1, metadata2]))

{'MAP': 0.7580690142192138,
 'MRR': 0.6666666666666666,
 'NDCG@5': 0.7234267649959281,
 'P@5': 0.8,
 'R@5': 0.13825757575757575,
 'F1@5': 0.23502722323049002,
 'NDCG@10': 0.8205228949141826,
 'P@10': 0.9,
 'R@10': 0.3181818181818182,
 'F1@10': 0.46785225718194257,
 'NDCG@20': 0.8338147007693439,
 'P@20': 0.875,
 'R@20': 0.6212121212121212,
 'F1@20': 0.7221269296740994,
 'avg_reranker_time_ms': 8818.163050018484,
 'avg_reranker_input_tokens': 11526.0}


# Storing result 

In [19]:
def save_results(ranked_output, metrics, model_path_or_name, threshold, k_values, output_dir="."):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_slug = model_path_or_name.replace("/", "_")
    
    metadata = {
        "model": model_path_or_name,
        "threshold": threshold,
        "k_values": k_values,
        "timestamp": timestamp,
        "n_queries": len(ranked_output)
    }

    results_payload = {
        "metadata": metadata,
        "metrics": metrics
    }

    metrics_path = f"{output_dir}/{model_slug}_{timestamp}_metrics.json"
    ranked_path = f"{output_dir}/{model_slug}_{timestamp}_ranked.json"

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(results_payload, f, indent=2)
    with open(ranked_path, "w", encoding="utf-8") as f:
        json.dump(ranked_output, f, indent=2)

    print(f"Metrics saved to {metrics_path}")
    print(f"Ranked output saved to {ranked_path}")

In [48]:
# alt save_results with fusing int a single file.
import json
from datetime import datetime

def save_results(
    scored: list[list[dict]],
    metadata: list[dict],
    metrics: dict,
    model_path_or_name: str,
    threshold: float,
    k_values: list[int],
    output_dir: str = "."
) -> None:
    """
    Saves ranked output and evaluation metrics to JSON files.

    Args:
        scored             : Output of rerank() calls — list of ranked candidate lists.
        metadata           : List of per-query metadata dicts from rerank().
        metrics            : Output of evaluate_reranker_dataset().
        model_path_or_name : Model identifier, used in filename.
        threshold          : Threshold used for binary label computation.
        k_values           : k values used in evaluation.
        output_dir         : Directory to write output files.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_slug = model_path_or_name.replace("/", "_")

    # Fuse scored and metadata by query index
    fused = [
        {
            "reranker_time_ms": meta["reranker_time_ms"],
            "reranker_input_tokens": meta["reranker_input_tokens"],
            "candidates": candidates
        }
        for candidates, meta in zip(scored, metadata)
    ]

    results_payload = {
        "metadata": {
            "model": model_path_or_name,
            "threshold": threshold,
            "k_values": k_values,
            "timestamp": timestamp,
            "n_queries": len(scored)
        },
        "metrics": metrics,
        "ranked_output": fused
    }

    output_path = f"{output_dir}/{model_slug}_{timestamp}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results_payload, f, indent=2)

    print(f"Results saved to {output_path}")

# Full pipeline

In [50]:
# alt pipeline
model, tokenizer = load_reranker(model_path_or_name="ncbi/MedCPT-Cross-Encoder")

train_dataset, val_dataset, test_dataset, abstracts = load_bioASQ(
    abstracts_path='../data/BioASQ-training14b/abstract_list_tokenized.json',
    bioasq_path='../data/BioASQ-training14b/training14b.json',
)

# prepare candidate 
# train_pool = build_candidate_pool(train_dataset, abstracts)
# test_pool = build_candidate_pool(test_dataset, abstracts)
# with open("train_pool.json", "r", encoding="utf-8") as train_p_json:
#     train_pool = json.load(train_p_json)
with open("test_pool.json", "r", encoding="utf-8") as test_p_json:
    test_pool = json.load(test_p_json)


queries = [test_pool[i]['query'] for i in range(10)]
candidates_list = [test_pool[i]['candidates'] for i in range(10)]

scoreTest = []
metadataTest = []
for q, cand in tqdm(zip(queries, candidates_list), total=len(queries), desc="Reranking"):
    ranked, meta = rerank(query=q, candidates=cand, model=model, tokenizer=tokenizer)
    scoreTest.append(ranked)
    metadataTest.append(meta)

threshold = 0.5
k_values=[5,10,15,20]
metrics = evaluate_reranker_dataset(scoreTest, metadataTest, threshold=threshold, k_values=k_values)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14334.50it/s]


Reranker loaded from 'ncbi/MedCPT-Cross-Encoder' on device 'cpu'.
Dataset loaded: 4582 train / 573 val / 573 test entries.
Entries dropped (no matching abstracts): 1


Reranking: 100%|██████████| 10/10 [01:13<00:00,  7.32s/it]


In [52]:
save_results(scoreTest, metadataTest, metrics, "ncbi/MedCPT-Cross-Encoder", threshold=threshold, k_values=k_values, output_dir="result/")

Results saved to result//ncbi_MedCPT-Cross-Encoder_20260528_161055.json


In [3]:
import sys
sys.path.append('../')

from data.load_dataset import load_bioASQ, build_candidate_pool_bioASQ
from models.load_reranker import load_reranker
from saveResult import save_results
from inference.rerank import rerank, evaluate_reranker_dataset
import json
import tqdm

# this might need to come from a .sh script later. 
modelName = "ncbi/MedCPT-Cross-Encoder"
threshold = 0.5
k_values=[5,10,15,20]
abstracts_path = "../data/BioASQ-training14b/abstract_list_tokenized.json"
bioasq_path='../data/BioASQ-training14b/training14b.json'

model, tokenizer = load_reranker(model_path_or_name=modelName)

train_dataset, val_dataset, test_dataset, abstracts = load_bioASQ(
    abstracts_path=abstracts_path,
    bioasq_path=bioasq_path,
)

# prepare candidate 
train_pool = [build_candidate_pool(entry, abstracts) for entry in train_dataset]
test_pool = [build_candidate_pool(entry, abstracts) for entry in test_dataset]
# with open("train_pool.json", "r", encoding="utf-8") as train_p_json:
#     train_pool = json.load(train_p_json)
# with open("test_pool.json", "r", encoding="utf-8") as test_p_json:
#     test_pool = json.load(test_p_json)

queries = [test_pool[i]['query'] for i in range(10)]
candidates_list = [test_pool[i]['candidates'] for i in range(10)]

scoreTest = []
for q, cand in tqdm(zip(queries, candidates_list), total=len(queries), desc="Reranking"):
    scoreTest.append(rerank(query=q, candidates=cand, model=model, tokenizer=tokenizer))

metrics = evaluate_reranker_dataset(scoreTest, threshold=threshold, k_values=k_values)

save_results(scoreTest, metrics=metrics, model_path_or_name=modelName, threshold=threshold, k_values=k_values, output_dir="result/TestBaseLineReranker")



Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1774.12it/s]


Reranker loaded from 'ncbi/MedCPT-Cross-Encoder' on device 'cpu'.
Dataset loaded: 4582 train / 573 val / 573 test entries.
Entries dropped (no matching abstracts): 1


TypeError: list indices must be integers or slices, not str

In [ ]:
import sys
sys.path.append('../')
# so that it sits at the root of the program. 

from data.load_dataset import load_bioASQ, build_candidate_pool_bioASQ
from models.load_reranker import load_reranker
from saveResult import save_results
from inference.rerank import rerank, evaluate_reranker_dataset
import json
from tqdm import tqdm

# those might need to come from a .sh script later. 
modelName = "ncbi/MedCPT-Cross-Encoder"
threshold = 0.5 # % of overlap a sentence must have with a snippet to be considered a good sentence. 
k_values=[5,10,15,20]
abstracts_path = "../data/BioASQ-training14b/abstract_list_tokenized.json"
bioasq_path='../data/BioASQ-training14b/training14b.json'
train_ratio = 0.8
test_ratio = 0.1
val_ratio = 0.1
batch_size = 16 # how many sentences the reranker treast at once for a given query. 
output_dir="result/TestBaseLineReranker"

model, tokenizer = load_reranker(model_path_or_name=modelName)

train_dataset, val_dataset, test_dataset, abstracts = load_bioASQ(
    abstracts_path=abstracts_path,
    bioasq_path=bioasq_path,
    train_ratio=train_ratio, 
    test_ratio=test_ratio,
    val_ratio=val_ratio
)

# prepare candidate 
# train_pool = [build_candidate_pool_bioASQ(entry, abstracts) for entry in tqdm(train_dataset, desc="Building train pool")]
# test_pool = [build_candidate_pool_bioASQ(entry, abstracts) for entry in tqdm(test_dataset, desc="Building test pool")]

# if things were already run before, this can be used more quickly. 
# with open("train_pool.json", "r", encoding="utf-8") as train_p_json:
#     train_pool = json.load(train_p_json)
with open("test_pool.json", "r", encoding="utf-8") as test_p_json:
    test_pool = json.load(test_p_json)

queries = [test_pool[i]['query'] for i in range(10)]
candidates_list = [test_pool[i]['candidates'] for i in range(10)]

# queries = [test_pool[i]['query'] for i in range(len(test_pool))]
# candidates_list = [test_pool[i]['candidates'] for i in range(len(test_pool))]

scoreTest = []
metadataTest = []
for q, cand in tqdm(zip(queries, candidates_list), total=len(queries), desc="Reranking"):
    ranked, meta = rerank(query=q, candidates=cand, model=model, tokenizer=tokenizer)
    scoreTest.append(ranked)
    metadataTest.append(meta)

# potentially, this might get looped over threshold values. 
metrics = evaluate_reranker_dataset(scoreTest, threshold=threshold, k_values=k_values)
save_results(ranked_output=scoreTest, metrics=metrics, model_path_or_name=modelName, threshold=threshold, k_values=k_values, output_dir=output_dir)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28573.30it/s]


Reranker loaded from 'ncbi/MedCPT-Cross-Encoder' on device 'cpu'.
Dataset loaded: 4582 train / 573 val / 573 test entries.
Entries dropped (no matching abstracts): 1


Reranking: 100%|██████████| 10/10 [00:55<00:00,  5.54s/it]

Metrics saved to result\TestBaseLineReranker\ncbi_MedCPT-Cross-Encoder_20260528_154058_metrics.json
Ranked output saved to result\TestBaseLineReranker\ncbi_MedCPT-Cross-Encoder_20260528_154058_ranked.json
